In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Train/Test
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Optuna (para después)
import optuna
from optuna.samplers import TPESampler

# Otros
from datetime import datetime

# ── Constantes globales ──
RANDOM_STATE = 42
TEST_SIZE    = 0.2

In [2]:
# Cargar dataset
df = pd.read_csv("../data/raw/stroke_dataset.csv")

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
print(df.head(3))

Dataset cargado: 4981 filas, 11 columnas

Primeras filas:
   gender   age  hypertension  heart_disease ever_married work_type  \
0    Male  67.0             0              1          Yes   Private   
1    Male  80.0             0              1          Yes   Private   
2  Female  49.0             0              0          Yes   Private   

  Residence_type  avg_glucose_level   bmi   smoking_status  stroke  
0          Urban             228.69  36.6  formerly smoked       1  
1          Rural             105.92  32.5     never smoked       1  
2          Urban             171.23  34.4           smokes       1  


In [ ]:
import os
# from dotenv import load_dotenv
import os
import mlflow

# cargar variables del .env
# load_dotenv()

# # usar variables
# mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
# mlflow.set_experiment(os.getenv("MLFLOW_EXPERIMENT"))

# print("Tracking URI:", mlflow.get_tracking_uri())




# #MLflow está en la raíz del proyecto
# mlruns_path = os.path.abspath("../mlruns")

# #Decirle a MLflow dónde guardar
# mlflow.set_tracking_uri(f"file:///{mlruns_path}")

# print(f"✓ MLflow usando carpeta compartida: {mlruns_path}")

import os
import mlflow

# 1️ Primero definir dónde guardar (MUY IMPORTANTE)
mlruns_path = os.path.abspath("../mlruns")
mlflow.set_tracking_uri(f"file:///{mlruns_path}")

# 2️ Luego definir el experimento
mlflow.set_experiment("Ictus_Project")
# mlflow.set_experiment("stroke_project")

print("Tracking URI:", mlflow.get_tracking_uri())


Traceback (most recent call last):
  File "c:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 329, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 427, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1373, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\.venv\Lib\sit

Tracking URI: file:///c:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\mlruns


In [5]:
def get_dataset(version="full"):
    """
    Obtiene el dataset en diferentes versiones
    
    version="full": todos los registros + columna 'group'
    version="adults": solo mayores de 16 años (sin columna 'group')
    """
    
    df_copy = df.copy()
    
    # LIMPIEZA 1: menores de 16 con smoking_status Unknown → not_applied
    df_copy.loc[(df_copy['age'] < 17) & (df_copy['smoking_status'] == 'Unknown'), 'smoking_status'] = 'not_applied'
    
    # LIMPIEZA 2: work_type='children' → 'not_applied'
    df_copy.loc[df_copy['work_type'] == 'children', 'work_type'] = 'not_applied'
    
    if version == "full":
        # Crear columna group: 'children' o 'adults'
        df_copy['group'] = np.where(df_copy['age'] < 17, 'children', 'adults')
        return df_copy
    
    elif version == "adults":
        # Filtrar solo adultos (age > 16)
        df_copy = df_copy[df_copy['age'] > 16].copy()
        # NO añadimos columna group en esta versión
        return df_copy

##  Función train_model_with_pipeline()

### ¿Qué hace?
Función que **entrena automáticamente un modelo** usando un Pipeline.

### Los 3 pasos internos:
1. **OneHot Encoding**: Convierte texto (gender, work_type, etc.) en números (0/1)
2. **SMOTE**: Crea datos sintéticos de la clase minoritaria (stroke=1) SOLO en train
3. **Entrenar modelo**: Entrena el modelo (Logistic o XGBoost)

### Entrada (Input):
- `X_train, X_test`: Features de entrenamiento y test
- `y_train, y_test`: Targets de entrenamiento y test
- `model`: El modelo a entrenar (ej: LogisticRegression())
- `model_name`: Nombre del modelo para MLflow (ej: "LogisticRegression")
- `dataset_version`: "full" o "adults"

### Salida (Output):
- `pipeline`: El modelo entrenado + procesamiento
- `acc, prec, rec, f1, auc`: Las 5 métricas principales

### ¿Por qué Pipeline?
Garantiza que los datos pasen por los MISMOS pasos en train Y test.
SMOTE solo actúa en entrenamiento, no en predicción (evita data leakage).

### Ejemplo de uso:
```python
pipeline, acc, prec, rec, f1, auc = train_model_with_pipeline(
    X_train, X_test, y_train, y_test,
    LogisticRegression(),
    "LogisticRegression",
    "full"
)
```

In [6]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# FUNCIÓN 1: Para Logistic Regression (CON Scaling)
def train_logistic_with_pipeline(X_train, X_test, y_train, y_test, model, dataset_version="full"):
    """
    Entrena Logistic Regression con Pipeline (OneHot + Scaling + SMOTE + Modelo)
    """
    
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    num_cols = X_train.select_dtypes(exclude='object').columns.tolist()
    
    # Preprocesador CON Scaling para numéricas
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)  #  Scaling aquí
    ])
    
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', model)
    ])
    
    print(f"\n=== Logistic Regression | {dataset_version} ===")
    pipeline.fit(X_train, y_train)
    
  
    # Predicciones en TRAIN y TEST
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Métricas en TRAIN
    acc_train = accuracy_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train, average='weighted')

    # Métricas en TEST
    acc_test = accuracy_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # Diferencias (para detectar overfitting)
    diff_acc = abs(acc_train - acc_test)
    diff_f1 = abs(f1_train - f1_test)

    print(f"Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f} | Diff: {diff_acc:.4f}")
    print(f"Train F1:  {f1_train:.4f} | Test F1:  {f1_test:.4f} | Diff: {diff_f1:.4f}")

    if diff_acc < 0.05:
        print("✓ Sin overfitting")
    else:
        print("⚠ Posible overfitting")
        
    
    acc = acc_test  # Ya está calculado
    prec = precision_score(y_test, y_pred_test, average='weighted')
    rec = recall_score(y_test, y_pred_test, average='weighted')
    f1 = f1_test  # Ya está calculado
    auc = roc_auc_score(y_test, y_prob)
        
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    
    return pipeline, acc, prec, rec, f1, auc, diff_acc, diff_f1


In [7]:
# FUNCIÓN 2: Para XGBoost (SIN Scaling)
def train_xgboost_with_pipeline(X_train, X_test, y_train, y_test, model, dataset_version="full"):
    """
    Entrena XGBoost con Pipeline (OneHot + SMOTE + Modelo, sin Scaling)
    """
    
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    num_cols = X_train.select_dtypes(exclude='object').columns.tolist()
    
    # Preprocesador SIN Scaling para numéricas
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)  # ✗ Sin scaling
    ])
    
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', model)
    ])
    
    print(f"\n=== XGBoost | {dataset_version} ===")
    pipeline.fit(X_train, y_train)
    
  
    
    # Predicciones en TRAIN y TEST
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Métricas en TRAIN
    acc_train = accuracy_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train, average='weighted')

    # Métricas en TEST
    acc_test = accuracy_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # Diferencias (para detectar overfitting)
    diff_acc = abs(acc_train - acc_test)
    diff_f1 = abs(f1_train - f1_test)

    print(f"Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f} | Diff: {diff_acc:.4f}")
    print(f"Train F1:  {f1_train:.4f} | Test F1:  {f1_test:.4f} | Diff: {diff_f1:.4f}")

    if diff_acc < 0.05:
        print("✓ Sin overfitting")
    else:
        print("⚠ Posible overfitting")
        
    
    acc = acc_test  # Ya está calculado
    prec = precision_score(y_test, y_pred_test, average='weighted')
    rec = recall_score(y_test, y_pred_test, average='weighted')
    f1 = f1_test  # Ya está calculado
    auc = roc_auc_score(y_test, y_prob)
        
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    
    return pipeline, acc, prec, rec, f1, auc, diff_acc, diff_f1

### CREAMOS EL MODELO REGRESION LOGISTICA

In [8]:
# Crear el modelo Logistic Regression CON Ridge Loss
log_model = LogisticRegression(
    class_weight='balanced',
    penalty='l2',              # ← Ridge Loss (L2)
    C=1.0,                     # Parámetro de regularización (menor C = más regularización)
    max_iter=1000,
    random_state=RANDOM_STATE
)

print("Modelo Logistic Regression (con Ridge Loss L2) creado ✓")

Modelo Logistic Regression (con Ridge Loss L2) creado ✓


### PREPARAMOS PARA ENTRENARCON REGRESION LOGISTICA CON EL DATASET FULL (el que tiene columnas separadas niños y adultos)

In [9]:
# Obtener dataset FULL (con niños)
df_full = get_dataset(version="full")

print(f"Dataset FULL cargado: {df_full.shape[0]} filas")
print(f"Columnas: {df_full.columns.tolist()}")

# Separar X e y
X_full = df_full.drop('stroke', axis=1)
y_full = df_full['stroke']

# Split train/test
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print(f"\nDataset FULL dividido:")
print(f"  X_train: {X_train_full.shape}")
print(f"  X_test: {X_test_full.shape}")

Dataset FULL cargado: 4981 filas
Columnas: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke', 'group']

Dataset FULL dividido:
  X_train: (3984, 11)
  X_test: (997, 11)


In [10]:
# Entrenar Logistic Regression con dataset FULL
pipeline_log_full, acc_log_full, prec_log_full, rec_log_full, f1_log_full, auc_log_full, diff_acc_log_full, diff_f1_log_full = train_logistic_with_pipeline(
    X_train_full, X_test_full, y_train_full, y_test_full,
    log_model,
    dataset_version="full"
)

print(f"\n✓ Logistic Regression (FULL) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_log_full:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_log_full:.4f}")


=== Logistic Regression | full ===
Train Acc: 0.7432 | Test Acc: 0.7482 | Diff: 0.0050
Train F1:  0.8154 | Test F1:  0.8185 | Diff: 0.0031
✓ Sin overfitting
Accuracy: 0.7482 | Precision: 0.9423 | Recall: 0.7482 | F1: 0.8185 | AUC: 0.8372

✓ Logistic Regression (FULL) entrenado
  Overfitting - Diff Accuracy: 0.0050
  Overfitting - Diff F1: 0.0031


### HACEMOS MATRIZ DE CONFUSION

In [11]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
import os
from pathlib import Path

# Crear carpeta assets si no existe
BASE_DIR = Path().resolve().parent  # Sube de notebooks/ a raíz
assets_dir = BASE_DIR / "assets"
assets_dir.mkdir(exist_ok=True)  # Crea si no existe

# Hacer predicción
y_pred_log_full = pipeline_log_full.predict(X_test_full)

# Matriz de confusión
cm = confusion_matrix(y_test_full, y_pred_log_full)

# Gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - Logistic Regression (FULL)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_path = os.path.join(assets_dir, "confusion_matrix_lr_full_v2.png")
plt.savefig(cm_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz guardada en: {cm_path}")

✓ Matriz guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_lr_full_v2.png


### USAMOS MLFLOW

In [ ]:
# Registrar en MLflow - Logistic Regression FULL
# mlflow.set_experiment("stroke_project")

with mlflow.start_run(run_name="LogisticRegression_full_v2"):
    
    # Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("author", "Gema")
    
    # Métricas 
    mlflow.log_metric("accuracy", round(acc_log_full, 4))
    mlflow.log_metric("precision", round(prec_log_full, 4))
    mlflow.log_metric("recall", round(rec_log_full, 4))
    mlflow.log_metric("f1", round(f1_log_full, 4))
    mlflow.log_metric("auc", round(auc_log_full, 4))
    
    # Métricas OVERFITTING (NUEVO)
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_log_full, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_log_full, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_path, artifact_path="confusion_matrices")
    
    # Modelo
    # mlflow.sklearn.log_model(pipeline_log_full, "modelo_logistic") de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    print("✓ LogisticRegression FULL registrado en MLflow")

mlflow.end_run()

✓ LogisticRegression FULL registrado en MLflow
🏃 View run LogisticRegression_full_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/00a5ba224b094408bd576d9469d8e84b
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


### CREAMOS EL MODELO XGBOOST

In [12]:
# Crear el modelo XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    scale_pos_weight=19,  # Porque stroke es minoritaria (95% vs 5%)
    eval_metric='logloss'
)

print("Modelo XGBoost creado ✓")

Modelo XGBoost creado ✓


### PREPARAMOS PARA ENTRENARCON XGBOOST CON EL DATASET FULL (el que tiene columnas separadas niños y adultos)

In [13]:
# Obtener dataset FULL (igual que con Logistic)
df_full = get_dataset(version="full")

# Separar X e y
X_full = df_full.drop('stroke', axis=1)
y_full = df_full['stroke']

# Split train/test
X_train_full_xgb, X_test_full_xgb, y_train_full_xgb, y_test_full_xgb = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print(f"Dataset FULL para XGBoost:")
print(f"  X_train: {X_train_full_xgb.shape}")
print(f"  X_test: {X_test_full_xgb.shape}")

Dataset FULL para XGBoost:
  X_train: (3984, 11)
  X_test: (997, 11)


In [14]:
pipeline_xgb_full, acc_xgb_full, prec_xgb_full, rec_xgb_full, f1_xgb_full, auc_xgb_full, diff_acc_xgb_full, diff_f1_xgb_full = train_xgboost_with_pipeline(
    X_train_full_xgb, X_test_full_xgb, y_train_full_xgb, y_test_full_xgb,
    xgb_model,
    dataset_version="full"
)

print(f"\n✓ XGBoost (FULL) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_xgb_full:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_xgb_full:.4f}")


=== XGBoost | full ===
Train Acc: 0.9352 | Test Acc: 0.8786 | Diff: 0.0566
Train F1:  0.9469 | Test F1:  0.8992 | Diff: 0.0477
⚠ Posible overfitting
Accuracy: 0.8786 | Precision: 0.9254 | Recall: 0.8786 | F1: 0.8992 | AUC: 0.8123

✓ XGBoost (FULL) entrenado
  Overfitting - Diff Accuracy: 0.0566
  Overfitting - Diff F1: 0.0477


### HACEMOS MATRIZ DE CONFUSION

In [15]:
# Hacer predicción
y_pred_xgb_full = pipeline_xgb_full.predict(X_test_full_xgb)
# Matriz de confusión
cm_xgb_full = confusion_matrix(y_test_full_xgb, y_pred_xgb_full)
# Gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb_full, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - XGBoost (FULL)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_xgb_path = os.path.join(assets_dir, "confusion_matrix_xgb_full_v2.png")
plt.savefig(cm_xgb_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz XGBoost guardada en: {cm_xgb_path}")


✓ Matriz XGBoost guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_full_v2.png


### REGISTRAMOS EN MLFLOW

In [16]:
# Registrar en MLflow - XGBoost FULL (métricas + gráfico)
with mlflow.start_run(run_name="XGBoost_full_v2"):
    
    # Parámetros
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", 19)
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_full, 4))
    mlflow.log_metric("precision", round(prec_xgb_full, 4))
    mlflow.log_metric("recall", round(rec_xgb_full, 4))
    mlflow.log_metric("f1", round(f1_xgb_full, 4))
    mlflow.log_metric("auc", round(auc_xgb_full, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_full, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_full, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_path, artifact_path="confusion_matrices")
    
     # Modelo
    # mlflow.sklearn.log_model(pipeline_xgb_full, "modelo_xgboost") de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    
    print("✓ XGBoost FULL registrado (métricas + gráfico)")

mlflow.end_run()

✓ XGBoost FULL registrado (métricas + gráfico)
🏃 View run XGBoost_full_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/e224502a71a849b680dff5bc60f6cf87
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


### CARGAMOS DATASET ADULTOS Y PREPARAMOS PARA EL ENTRENAMIENTO

In [16]:
df_adults = get_dataset(version="adults")
print(f"Dataset ADULTS cargado: {df_adults.shape[0]} filas")
print(f"Columnas: {df_adults.columns.tolist()}")

#separar X e y
X_adults = df_adults.drop('stroke', axis=1) 
y_adults = df_adults['stroke']

# Split train/test
X_train_adults, X_test_adults, y_train_adults, y_test_adults = train_test_split(
    X_adults, y_adults,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_adults
)

print(f"\nDataset ADULTS dividido:")
print(f"  X_train: {X_train_adults.shape}") 

Dataset ADULTS cargado: 4211 filas
Columnas: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']

Dataset ADULTS dividido:
  X_train: (3368, 10)


### ENTRENAMOS CON REGRESION LOGISTICA

In [17]:
pipeline_log_adults, acc_log_adults, prec_log_adults, rec_log_adults, f1_log_adults, auc_log_adults, diff_acc_log_adults, diff_f1_log_adults = train_logistic_with_pipeline(
    X_train_adults, X_test_adults, y_train_adults, y_test_adults,
    log_model,  
    dataset_version="adults"
)   
print(f"\n✓ Logistic Regression (ADULTS) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_log_adults:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_log_adults:.4f}")


=== Logistic Regression | adults ===
Train Acc: 0.7221 | Test Acc: 0.7082 | Diff: 0.0139
Train F1:  0.7955 | Test F1:  0.7857 | Diff: 0.0098
✓ Sin overfitting
Accuracy: 0.7082 | Precision: 0.9335 | Recall: 0.7082 | F1: 0.7857 | AUC: 0.8389

✓ Logistic Regression (ADULTS) entrenado
  Overfitting - Diff Accuracy: 0.0139
  Overfitting - Diff F1: 0.0098


### HACEMOS MATRIZ DE CONFUSION

In [19]:
y_pred_log_adults = pipeline_log_adults.predict(X_test_adults)
cm_log_adults = confusion_matrix(y_test_adults, y_pred_log_adults)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_log_adults, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - Logistic Regression (ADULTS)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_log_path = os.path.join(assets_dir, "confusion_matrix_log_adults_v2.png")
plt.savefig(cm_log_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz Logistic Regression guardada en: {cm_log_path}")

✓ Matriz Logistic Regression guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_log_adults_v2.png


### REGISTRAMOS EN MLFLOW

In [20]:
with mlflow.start_run(run_name="LogisticRegression_adults_v2"):
    
    # Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("dataset", "adults")
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_log_adults, 4))
    mlflow.log_metric("precision", round(prec_log_adults, 4))
    mlflow.log_metric("recall", round(rec_log_adults, 4))
    mlflow.log_metric("f1", round(f1_log_adults, 4))
    mlflow.log_metric("auc", round(auc_log_adults, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_log_adults, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_log_adults, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_log_path, artifact_path="confusion_matrices")
    
     # Modelo
    # mlflow.sklearn.log_model(pipeline_log_adults, "modelo_logistic_adults") de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    
    print("✓ Logistic Regression ADULTS registrado (métricas + gráfico)")

✓ Logistic Regression ADULTS registrado (métricas + gráfico)
🏃 View run LogisticRegression_adults_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/caa426ac469f4484abfa6225618a2b53
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


### ENTRENAMOS CON XGBOST

In [18]:
pipeline_xgboost_adults, acc_xgb_adults, prec_xgb_adults, rec_xgb_adults, f1_xgb_adults, auc_xgb_adults, diff_acc_xgb_adults, diff_f1_xgb_adults = train_xgboost_with_pipeline(
    X_train_adults, X_test_adults, y_train_adults, y_test_adults,
    xgb_model,
    dataset_version="adults")

print(f"\n✓ XGBoost (ADULTS) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_xgb_adults:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_xgb_adults:.4f}")


=== XGBoost | adults ===
Train Acc: 0.9121 | Test Acc: 0.8458 | Diff: 0.0663
Train F1:  0.9288 | Test F1:  0.8752 | Diff: 0.0536
⚠ Posible overfitting
Accuracy: 0.8458 | Precision: 0.9152 | Recall: 0.8458 | F1: 0.8752 | AUC: 0.7813

✓ XGBoost (ADULTS) entrenado
  Overfitting - Diff Accuracy: 0.0663
  Overfitting - Diff F1: 0.0536


### HACEMOS MATRIZ DE CONFUSION

In [22]:
y_pred_xgb_adults = pipeline_xgboost_adults.predict(X_test_adults)
cm_xgb_adults = confusion_matrix(y_test_adults, y_pred_xgb_adults)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb_adults, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - XBoost (ADULTS)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_xgb_path = os.path.join(assets_dir, "confusion_matrix_xgb_adults_v2.png")
plt.savefig(cm_xgb_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz XBoost guardada en: {cm_xgb_path}")



✓ Matriz XBoost guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_adults_v2.png


### REGISTRAMOS EN MLFLOW

In [23]:
with mlflow.start_run(run_name="XGBoost_adults_v2"):
    # Parámetros
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "adults")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", 19)
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_adults, 4))
    mlflow.log_metric("precision", round(prec_xgb_adults, 4))
    mlflow.log_metric("recall", round(rec_xgb_adults, 4))
    mlflow.log_metric("f1", round(f1_xgb_adults, 4))
    mlflow.log_metric("auc", round(auc_xgb_adults, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_adults, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_adults, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_path, artifact_path="confusion_matrices")
    
    #Modelo
    mlflow.sklearn.log_model(pipeline_xgboost_adults, "modelo_xgboost_adults") #de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    
    print("✓ XGBoost ADULTS registrado (métricas + gráfico)")

2026/04/21 13:09:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/21 13:09:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ XGBoost ADULTS registrado (métricas + gráfico)
🏃 View run XGBoost_adults_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/b22576637ffe4cf396defe44a4868e2b
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


### UTILIZAMOS OPTUNA PARA REGRESION LOGIGISTICA DATASET PARA LAS DOS VERSIONES 

In [19]:
def objective_logistic_generic(trial, X_train, X_test, y_train, y_test, dataset_version):
    """
    Función objetivo GENÉRICA para Optuna - Logistic Regression
    Funciona para FULL o ADULTS
    """
    
    # Parámetros a OPTIMIZAR
    C = trial.suggest_float('C', 0.001, 10.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l2', 'l1'])
    
    # Crear modelo
    log_model_trial = LogisticRegression(
        C=C,
        penalty=penalty,
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver='saga'
    )
    
    # Entrenar con la función
    _, acc, prec, rec, f1, auc, diff_acc, diff_f1 = train_logistic_with_pipeline(
        X_train, X_test, y_train, y_test,
        log_model_trial,
        dataset_version=dataset_version
    )
    
    return f1

### OPTIMIZAMOS PARA DATASET FULL

In [20]:
# Optimizar FULL
study_logistic_full = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

print("🔍 Optimizando Logistic Regression (FULL)...")
study_logistic_full.optimize(
    lambda trial: objective_logistic_generic(trial, X_train_full, X_test_full, y_train_full, y_test_full, "full"),
    n_trials=20,
    show_progress_bar=True
)

best_params_logistic_full = study_logistic_full.best_trial.params
print(f"✓ Mejores parámetros FULL: {best_params_logistic_full}")

[I 2026-04-22 12:46:52,104] A new study created in memory with name: no-name-d0f905fd-f484-418f-a322-81efad59c396


🔍 Optimizando Logistic Regression (FULL)...


Best trial: 0. Best value: 0.817795:   5%|▌         | 1/20 [00:00<00:02,  7.96it/s]


=== Logistic Regression | full ===
Train Acc: 0.7425 | Test Acc: 0.7472 | Diff: 0.0048
Train F1:  0.8149 | Test F1:  0.8178 | Diff: 0.0029
✓ Sin overfitting
Accuracy: 0.7472 | Precision: 0.9409 | Recall: 0.7472 | F1: 0.8178 | AUC: 0.8384
[I 2026-04-22 12:46:52,232] Trial 0 finished with value: 0.8177950535352727 and parameters: {'C': 0.03148911647956861, 'penalty': 'l2'}. Best is trial 0 with value: 0.8177950535352727.

=== Logistic Regression | full ===


Best trial: 1. Best value: 0.818549:  10%|█         | 2/20 [00:00<00:03,  5.32it/s]

...

Train Acc: 0.7432 | Test Acc: 0.7482 | Diff: 0.0050
Train F1:  0.8154 | Test F1:  0.8185 | Diff: 0.0031
✓ Sin overfitting
Accuracy: 0.7482 | Precision: 0.9423 | Recall: 0.7482 | F1: 0.8185 | AUC: 0.8373
[I 2026-04-22 12:46:57,025] Trial 17 finished with value: 0.8185487887487504 and parameters: {'C': 0.2592416433300666, 'penalty': 'l2'}. Best is trial 1 with value: 0.8185487887487504.

=== Logistic Regression | full ===


Best trial: 1. Best value: 0.818549:  95%|█████████▌| 19/20 [00:05<00:00,  2.66it/s]

Train Acc: 0.7435 | Test Acc: 0.7482 | Diff: 0.0048
Train F1:  0.8156 | Test F1:  0.8185 | Diff: 0.0030
✓ Sin overfitting
Accuracy: 0.7482 | Precision: 0.9423 | Recall: 0.7482 | F1: 0.8185 | AUC: 0.8372
[I 2026-04-22 12:46:57,609] Trial 18 finished with value: 0.8185487887487504 and parameters: {'C': 1.6261036461649985, 'penalty': 'l2'}. Best is trial 1 with value: 0.8185487887487504.

=== Logistic Regression | full ===


Best trial: 1. Best value: 0.818549: 100%|██████████| 20/20 [00:05<00:00,  3.44it/s]

Train Acc: 0.7430 | Test Acc: 0.7482 | Diff: 0.0053
Train F1:  0.8152 | Test F1:  0.8185 | Diff: 0.0033
✓ Sin overfitting
Accuracy: 0.7482 | Precision: 0.9423 | Recall: 0.7482 | F1: 0.8185 | AUC: 0.8371
[I 2026-04-22 12:46:57,917] Trial 19 finished with value: 0.8185487887487504 and parameters: {'C': 1.1322339023737755, 'penalty': 'l1'}. Best is trial 1 with value: 0.8185487887487504.
✓ Mejores parámetros FULL: {'C': 0.24810409748678125, 'penalty': 'l2'}


### ENTRENAMOS REGRESION LOGISTICA CON MEJORES PARAMETROS CON DATASET FULL

In [21]:
# Entrenar Logistic Regression FULL con MEJORES PARÁMETROS (optimizados por Optuna)
log_model_optimized = LogisticRegression(
    C=best_params_logistic_full['C'], # best_param_logistic_full esta guardado los mejores parametros encontrados por Optuna
    penalty=best_params_logistic_full['penalty'],
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='saga'
)

pipeline_log_full_opt, acc_log_full_opt, prec_log_full_opt, rec_log_full_opt, f1_log_full_opt, auc_log_full_opt, diff_acc_log_full_opt, diff_f1_log_full_opt = train_logistic_with_pipeline(
    X_train_full, X_test_full, y_train_full, y_test_full,
    log_model_optimized,
    dataset_version="full"
)

print(f"\n✓ Logistic Regression FULL (OPTIMIZADO) entrenado")


=== Logistic Regression | full ===
Train Acc: 0.7432 | Test Acc: 0.7482 | Diff: 0.0050
Train F1:  0.8154 | Test F1:  0.8185 | Diff: 0.0031
✓ Sin overfitting
Accuracy: 0.7482 | Precision: 0.9423 | Recall: 0.7482 | F1: 0.8185 | AUC: 0.8374

✓ Logistic Regression FULL (OPTIMIZADO) entrenado


### HACEMOS MATRIZ DE CONFUSION

In [22]:
y_pred_log_full_opt = pipeline_log_full_opt.predict(X_test_full)
cm_log_full_opt = confusion_matrix(y_test_full, y_pred_log_full_opt)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_log_full_opt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],            
            yticklabels=['No Stroke', 'Stroke'])        

plt.title('Matriz de Confusión - Logistic Regression FULL (OPTIMIZADO)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_log_full_opt_path = os.path.join(assets_dir, "confusion_matrix_log_full_opt.png")
plt.savefig(cm_log_full_opt_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz Logistic Regression FULL (OPTIMIZADO) guardada en: {cm_log_full_opt_path}")


✓ Matriz Logistic Regression FULL (OPTIMIZADO) guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_log_full_opt.png


### REGISTRAMOS EN MLFLOW

In [25]:
with mlflow.start_run(run_name="LogisticRegression_full_optimized_v2"):
    
    # Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("penalty", best_params_logistic_full['penalty'])
    mlflow.log_param("C", best_params_logistic_full['C'])
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_log_full_opt, 4))
    mlflow.log_metric("precision", round(prec_log_full_opt, 4))
    mlflow.log_metric("recall", round(rec_log_full_opt, 4))
    mlflow.log_metric("f1", round(f1_log_full_opt, 4))
    mlflow.log_metric("auc", round(auc_log_full_opt, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_log_full_opt, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_log_full_opt, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_log_full_opt_path, artifact_path="confusion_matrices")
    
    # Modelo
    mlflow.sklearn.log_model(pipeline_log_full_opt, "modelo_logistic_full_optimized") 
    
    
    
    print("✓ Logistic Regression FULL OPTIMIZADO registrado (métricas + gráfico)")

2026/04/22 12:54:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/22 12:54:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ Logistic Regression FULL OPTIMIZADO registrado (métricas + gráfico)


### OPTIMIZAMOS PARA DATASET ADULTS

In [29]:
# Optimizar ADULTS (MISMO código, diferente dataset)
study_logistic_adults = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

print("🔍 Optimizando Logistic Regression (ADULTS)...")
study_logistic_adults.optimize(
    lambda trial: objective_logistic_generic(trial, X_train_adults, X_test_adults, y_train_adults, y_test_adults, "adults"),# aqui nombrados el dataset ADULTS
    n_trials=20,
    show_progress_bar=True
)

best_params_logistic_adults = study_logistic_adults.best_trial.params
print(f"✓ Mejores parámetros ADULTS: {best_params_logistic_adults}")

[I 2026-04-21 13:09:19,289] A new study created in memory with name: no-name-bf619df8-7dd3-4028-bf51-adb6ac47a699


🔍 Optimizando Logistic Regression (ADULTS)...


Best trial: 0. Best value: 0.790923:   0%|          | 0/20 [00:00<?, ?it/s]


=== Logistic Regression | adults ===
Train Acc: 0.7227 | Test Acc: 0.7153 | Diff: 0.0074
Train F1:  0.7960 | Test F1:  0.7909 | Diff: 0.0051
✓ Sin overfitting
Accuracy: 0.7153 | Precision: 0.9356 | Recall: 0.7153 | F1: 0.7909 | AUC: 0.8393
[I 2026-04-21 13:09:19,374] Trial 0 finished with value: 0.7909226616924495 and parameters: {'C': 0.03148911647956861, 'penalty': 'l2'}. Best is trial 0 with value: 0.7909226616924495.

=== Logistic Regression | adults ===


Best trial: 2. Best value: 0.793449:  20%|██        | 4/20 [00:00<00:01, 10.01it/s]

...

Train Acc: 0.7162 | Test Acc: 0.7260 | Diff: 0.0098
Train F1:  0.7913 | Test F1:  0.7985 | Diff: 0.0073
✓ Sin overfitting
Accuracy: 0.7260 | Precision: 0.9344 | Recall: 0.7260 | F1: 0.7985 | AUC: 0.8393
[I 2026-04-21 13:09:21,026] Trial 14 finished with value: 0.7985417259290924 and parameters: {'C': 0.021836383093975134, 'penalty': 'l1'}. Best is trial 12 with value: 0.7993875347329991.

=== Logistic Regression | adults ===
Train Acc: 0.6865 | Test Acc: 0.6963 | Diff: 0.0099
Train F1:  0.7696 | Test F1:  0.7771 | Diff: 0.0075
✓ Sin overfitting
Accuracy: 0.6963 | Precision: 0.9347 | Recall: 0.6963 | F1: 0.7771 | AUC: 0.8268
[I 2026-04-21 13:09:21,107] Trial 15 finished with value: 0.777106454022231 and parameters: {'C': 0.0012461633766308474, 'penalty': 'l1'}. Best is trial 12 with value: 0.7993875347329991.

=== Logistic Regression | adults ===
Train Acc: 0.7147 | Test Acc: 0.7236 | Diff: 0.0089
Train F1:  0.7902 | Test F1:  0.7968 | Diff: 0.0067
✓ Sin overfitting
Accuracy: 0.7236 | P

Best trial: 12. Best value: 0.799388:  95%|█████████▌| 19/20 [00:02<00:00, 10.89it/s]

Train Acc: 0.6995 | Test Acc: 0.7129 | Diff: 0.0134
Train F1:  0.7792 | Test F1:  0.7892 | Diff: 0.0100
✓ Sin overfitting
Accuracy: 0.7129 | Precision: 0.9355 | Recall: 0.7129 | F1: 0.7892 | AUC: 0.8304
[I 2026-04-21 13:09:21,280] Trial 17 finished with value: 0.7892087132877138 and parameters: {'C': 0.002959430671634822, 'penalty': 'l1'}. Best is trial 12 with value: 0.7993875347329991.

=== Logistic Regression | adults ===
Train Acc: 0.7230 | Test Acc: 0.7117 | Diff: 0.0112
Train F1:  0.7962 | Test F1:  0.7884 | Diff: 0.0078
✓ Sin overfitting
Accuracy: 0.7117 | Precision: 0.9355 | Recall: 0.7117 | F1: 0.7884 | AUC: 0.8399
[I 2026-04-21 13:09:21,368] Trial 18 finished with value: 0.7883503713431534 and parameters: {'C': 0.1964101424424174, 'penalty': 'l1'}. Best is trial 12 with value: 0.7993875347329991.

=== Logistic Regression | adults ===


Best trial: 12. Best value: 0.799388: 100%|██████████| 20/20 [00:02<00:00,  8.72it/s]

Train Acc: 0.7218 | Test Acc: 0.7082 | Diff: 0.0136
Train F1:  0.7953 | Test F1:  0.7857 | Diff: 0.0096
✓ Sin overfitting
Accuracy: 0.7082 | Precision: 0.9335 | Recall: 0.7082 | F1: 0.7857 | AUC: 0.8391
[I 2026-04-21 13:09:21,583] Trial 19 finished with value: 0.7857492344796446 and parameters: {'C': 3.5092247138920856, 'penalty': 'l1'}. Best is trial 12 with value: 0.7993875347329991.
✓ Mejores parámetros ADULTS: {'C': 0.0117894243971473, 'penalty': 'l1'}


### ENGRENAMOS REGRESION LOGISTICA CON LOS MEJORES PARAMETROS PARA DATASET ADULTOS

In [30]:
# Entrenar Logistic Regression ADULTS con MEJORES PARÁMETROS (optimizados por Optuna)
log_model_optimized_adults = LogisticRegression(
    C=best_params_logistic_adults['C'],
    penalty=best_params_logistic_adults['penalty'],
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='saga'
)

pipeline_log_adults_opt, acc_log_adults_opt, prec_log_adults_opt, rec_log_adults_opt, f1_log_adults_opt, auc_log_adults_opt, diff_acc_log_adults_opt, diff_f1_log_adults_opt = train_logistic_with_pipeline(
    X_train_adults, X_test_adults, y_train_adults, y_test_adults,
    log_model_optimized_adults,
    dataset_version="adults"
)

print(f"\n✓ Logistic Regression ADULTS (OPTIMIZADO) entrenado")
print(f"  Mejores parámetros: C={best_params_logistic_adults['C']:.4f}, penalty={best_params_logistic_adults['penalty']}")


=== Logistic Regression | adults ===
Train Acc: 0.7129 | Test Acc: 0.7272 | Diff: 0.0143
Train F1:  0.7889 | Test F1:  0.7994 | Diff: 0.0105
✓ Sin overfitting
Accuracy: 0.7272 | Precision: 0.9345 | Recall: 0.7272 | F1: 0.7994 | AUC: 0.8349

✓ Logistic Regression ADULTS (OPTIMIZADO) entrenado
  Mejores parámetros: C=0.0118, penalty=l1


### HACEMOS MATRIZ DE CONFUSION

In [31]:
y_pred_log_adults_opt= pipeline_log_adults_opt.predict(X_test_adults)
cm_log_adults_opt = confusion_matrix(y_test_adults, y_pred_log_adults_opt)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_log_adults_opt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],    
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - Logistic Regression ADULTS (OPTIMIZADO)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_log_adults_opt_path = os.path.join(assets_dir, "confusion_matrix_log_adults_opt.png")
plt.savefig(cm_log_adults_opt_path, dpi=100, bbox_inches='tight')       
plt.close() 


### REGSITRAMOS EN MLFLOW

In [32]:
with mlflow.start_run(run_name="LogisticRegression_adults_optimized_v2"):
    # Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("dataset", "adults")
    mlflow.log_param("penalty", best_params_logistic_adults['penalty'])
    mlflow.log_param("C", best_params_logistic_adults['C'])
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_log_adults_opt, 4))
    mlflow.log_metric("precision", round(prec_log_adults_opt, 4))
    mlflow.log_metric("recall", round(rec_log_adults_opt, 4))
    mlflow.log_metric("f1", round(f1_log_adults_opt, 4))
    mlflow.log_metric("auc", round(auc_log_adults_opt, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_log_adults_opt, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_log_adults_opt, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_log_adults_opt_path, artifact_path="confusion_matrices")
    
     # Modelo
    mlflow.sklearn.log_model(pipeline_log_adults_opt, "modelo_logistic_adults_optimized") 
    
    
    
    print("✓ Logistic Regression ADULTS OPTIMIZADO registrado (métricas + gráfico)")

2026/04/21 13:09:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/21 13:09:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ Logistic Regression ADULTS OPTIMIZADO registrado (métricas + gráfico)
🏃 View run LogisticRegression_adults_optimized_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/7fd2eff367744a4ea6b05bcbe62aecd1
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


### UTILIZAMOS OPTUNA PARA XGBOOST CON DATASET PARA LAS DOS VERSIONES 

In [26]:
def objective_xgboost_generic(trial, X_train, X_test, y_train, y_test, dataset_version):
    """
    Función objetivo GENÉRICA para Optuna - XGBoost
    Funciona para FULL o ADULTS
    """
    
    # Parámetros a OPTIMIZAR
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # Crear modelo
    xgb_model_trial = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=RANDOM_STATE,
        scale_pos_weight=19,
        eval_metric='logloss'
    )
    
    # Entrenar con la función
    _, acc, prec, rec, f1, auc, diff_acc, diff_f1 = train_xgboost_with_pipeline(
        X_train, X_test, y_train, y_test,
        xgb_model_trial,
        dataset_version=dataset_version
    )
    
    return f1

### OPTIMIZAMOS XGBOOST CON DATASET FULL

In [27]:
# Optimizar XGBoost FULL
study_xgb_full = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

print("🔍 Optimizando XGBoost (FULL)...")
study_xgb_full.optimize(
    lambda trial: objective_xgboost_generic(trial, X_train_full, X_test_full, y_train_full, y_test_full, "full"),
    n_trials=20,
    show_progress_bar=True
)

best_params_xgb_full = study_xgb_full.best_trial.params
print(f"\n✓ Mejores parámetros XGBoost FULL:")
for param, value in best_params_xgb_full.items():
    print(f"  {param}: {value}")

[I 2026-04-22 12:55:45,281] A new study created in memory with name: no-name-c6b3c6fe-e61c-4ff7-9319-b9be4d9634a1


🔍 Optimizando XGBoost (FULL)...


  0%|          | 0/20 [00:00<?, ?it/s]


=== XGBoost | full ===


Best trial: 0. Best value: 0.91798:   5%|▌         | 1/20 [00:00<00:13,  1.44it/s]

...

Train Acc: 1.0000 | Test Acc: 0.9258 | Diff: 0.0742
Train F1:  1.0000 | Test F1:  0.9180 | Diff: 0.0820
⚠ Posible overfitting
Accuracy: 0.9258 | Precision: 0.9108 | Recall: 0.9258 | F1: 0.9180 | AUC: 0.7890
[I 2026-04-22 12:55:55,419] Trial 17 finished with value: 0.9179803160885862 and parameters: {'n_estimators': 182, 'max_depth': 9, 'learning_rate': 0.21652870123936707}. Best is trial 2 with value: 0.9233327302563914.

=== XGBoost | full ===


Best trial: 2. Best value: 0.923333:  95%|█████████▌| 19/20 [00:11<00:00,  1.19it/s]

Train Acc: 1.0000 | Test Acc: 0.9278 | Diff: 0.0722
Train F1:  1.0000 | Test F1:  0.9191 | Diff: 0.0809
⚠ Posible overfitting
Accuracy: 0.9278 | Precision: 0.9114 | Recall: 0.9278 | F1: 0.9191 | AUC: 0.8127
[I 2026-04-22 12:55:56,503] Trial 18 finished with value: 0.9191335762774216 and parameters: {'n_estimators': 243, 'max_depth': 10, 'learning_rate': 0.13371460464045803}. Best is trial 2 with value: 0.9233327302563914.

=== XGBoost | full ===


Best trial: 2. Best value: 0.923333: 100%|██████████| 20/20 [00:11<00:00,  1.67it/s]

Train Acc: 0.9927 | Test Acc: 0.9188 | Diff: 0.0740
Train F1:  0.9930 | Test F1:  0.9176 | Diff: 0.0754
⚠ Posible overfitting
Accuracy: 0.9188 | Precision: 0.9164 | Recall: 0.9188 | F1: 0.9176 | AUC: 0.8013
[I 2026-04-22 12:55:57,224] Trial 19 finished with value: 0.9175641636690474 and parameters: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.08751956634681729}. Best is trial 2 with value: 0.9233327302563914.

✓ Mejores parámetros XGBoost FULL:
  n_estimators: 64
  max_depth: 9
  learning_rate: 0.18432335340553055


### ENTENAMOS  XGBOOST CON  LOS PARAMETROS OBTENIDOS CON EL DATASET FULL

In [28]:
# Entrenar XGBoost FULL con parámetros OPTIMIZADOS pero ajustados
xgb_model_optimized_full = xgb.XGBClassifier(
    n_estimators=best_params_xgb_full['n_estimators'],  
    max_depth=best_params_xgb_full['max_depth'],            
    learning_rate=best_params_xgb_full['learning_rate'], 
    random_state=RANDOM_STATE,
    scale_pos_weight=19,
    eval_metric='logloss'
)

pipeline_xgb_full_opt, acc_xgb_full_opt, prec_xgb_full_opt, rec_xgb_full_opt, f1_xgb_full_opt, auc_xgb_full_opt, diff_acc_xgb_full_opt, diff_f1_xgb_full_opt = train_xgboost_with_pipeline(
    X_train_full, X_test_full, y_train_full, y_test_full,
    xgb_model_optimized_full,
    dataset_version="full"
)


=== XGBoost | full ===
Train Acc: 0.9982 | Test Acc: 0.9268 | Diff: 0.0715
Train F1:  0.9983 | Test F1:  0.9233 | Diff: 0.0749
⚠ Posible overfitting
Accuracy: 0.9268 | Precision: 0.9201 | Recall: 0.9268 | F1: 0.9233 | AUC: 0.8141


### HACEMOS MATRIZ DE CONFUSION

In [36]:
y_pred_xgb_full_opt = pipeline_xgb_full_opt.predict(X_test_full)
cm_xgb_full_opt = confusion_matrix(y_test_full, y_pred_xgb_full_opt)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb_full_opt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],    
            yticklabels=['No Stroke', 'Stroke'])    
plt.title('Matriz de Confusión - XGBoost FULL (OPTIMIZADO)')
plt.ylabel('Actual')
plt.xlabel('Predicción')    
plt.tight_layout()
cm_xgb_full_opt_path = os.path.join(assets_dir, "confusion_matrix_xgb_full_opt.png")
plt.savefig(cm_xgb_full_opt_path, dpi=100, bbox_inches='tight') 
plt.close() 
print(f"✓ Matriz XGBoost FULL (OPTIMIZADO) guardada en: {cm_xgb_full_opt_path}")


✓ Matriz XGBoost FULL (OPTIMIZADO) guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_full_opt.png


### REGISTRAMOS EN MLFLOW

In [37]:
with mlflow.start_run(run_name="XGBoost_full_optimized_v2"):
    # Parámetros
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("n_estimators", best_params_xgb_full['n_estimators'])
    mlflow.log_param("max_depth", best_params_xgb_full['max_depth'])
    mlflow.log_param("learning_rate", best_params_xgb_full['learning_rate'])
    mlflow.log_param("scale_pos_weight", 19)
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_full_opt, 4))
    mlflow.log_metric("precision", round(prec_xgb_full_opt, 4))
    mlflow.log_metric("recall", round(rec_xgb_full_opt, 4))
    mlflow.log_metric("f1", round(f1_xgb_full_opt, 4))
    mlflow.log_metric("auc", round(auc_xgb_full_opt, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_full_opt, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_full_opt, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_full_opt_path, artifact_path="confusion_matrices")
    
     # Modelo
    mlflow.sklearn.log_model(pipeline_xgb_full_opt, "modelo_xgboost_full_optimized") 
    
    
    
    print("✓ XGBoost FULL OPTIMIZADO registrado (métricas + gráfico)")

    

2026/04/21 13:09:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/21 13:09:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ XGBoost FULL OPTIMIZADO registrado (métricas + gráfico)
🏃 View run XGBoost_full_optimized_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/61325d0c439e406dabc6f2f2048a5e72
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


### OPTIMIZAMOS XGBOOST CON DATASET ADULTS

In [38]:
# Optimizar XGBoost ADULTS
study_xgb_adults = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

print("🔍 Optimizando XGBoost (ADULTS)...")
study_xgb_adults.optimize(
    lambda trial: objective_xgboost_generic(trial, X_train_adults, X_test_adults, y_train_adults, y_test_adults, "adults"),
    n_trials=20,
    show_progress_bar=True
)

best_params_xgb_adults = study_xgb_adults.best_trial.params
print(f"\n✓ Mejores parámetros XGBoost ADULTS:")
for param, value in best_params_xgb_adults.items():
    print(f"  {param}: {value}")

[I 2026-04-21 13:09:48,832] A new study created in memory with name: no-name-52680bd7-9e83-4d87-ade7-25e1ce1f97df


🔍 Optimizando XGBoost (ADULTS)...


  0%|          | 0/20 [00:00<?, ?it/s]


=== XGBoost | adults ===


Best trial: 0. Best value: 0.907505:   5%|▌         | 1/20 [00:00<00:12,  1.50it/s]

...

Train Acc: 1.0000 | Test Acc: 0.9181 | Diff: 0.0819
Train F1:  1.0000 | Test F1:  0.9111 | Diff: 0.0889
⚠ Posible overfitting
Accuracy: 0.9181 | Precision: 0.9049 | Recall: 0.9181 | F1: 0.9111 | AUC: 0.7716
[I 2026-04-21 13:09:59,883] Trial 17 finished with value: 0.9111362899532721 and parameters: {'n_estimators': 197, 'max_depth': 9, 'learning_rate': 0.2220189526263523}. Best is trial 15 with value: 0.9170525498884793.

=== XGBoost | adults ===


Best trial: 15. Best value: 0.917053:  95%|█████████▌| 19/20 [00:11<00:00,  1.29it/s]

Train Acc: 1.0000 | Test Acc: 0.9122 | Diff: 0.0878
Train F1:  1.0000 | Test F1:  0.9075 | Diff: 0.0925
⚠ Posible overfitting
Accuracy: 0.9122 | Precision: 0.9031 | Recall: 0.9122 | F1: 0.9075 | AUC: 0.7502
[I 2026-04-21 13:10:00,600] Trial 18 finished with value: 0.9075053309249524 and parameters: {'n_estimators': 206, 'max_depth': 8, 'learning_rate': 0.23385957929045606}. Best is trial 15 with value: 0.9170525498884793.

=== XGBoost | adults ===


Best trial: 15. Best value: 0.917053: 100%|██████████| 20/20 [00:12<00:00,  1.59it/s]

Train Acc: 1.0000 | Test Acc: 0.9158 | Diff: 0.0842
Train F1:  1.0000 | Test F1:  0.9097 | Diff: 0.0903
⚠ Posible overfitting
Accuracy: 0.9158 | Precision: 0.9042 | Recall: 0.9158 | F1: 0.9097 | AUC: 0.7642
[I 2026-04-21 13:10:01,442] Trial 19 finished with value: 0.9096783458771734 and parameters: {'n_estimators': 257, 'max_depth': 9, 'learning_rate': 0.284989593323128}. Best is trial 15 with value: 0.9170525498884793.

✓ Mejores parámetros XGBoost ADULTS:
  n_estimators: 228
  max_depth: 10
  learning_rate: 0.1444463979732584


### ENTRENAMOS XGBOOST CON LOS PARAMETROS OBTENIDOS CON EL DATASET ADULTS

In [39]:
xgb_model_optimized_adults = xgb.XGBClassifier(
    n_estimators=best_params_xgb_adults['n_estimators'],  
    max_depth=best_params_xgb_adults['max_depth'],            
    learning_rate=best_params_xgb_adults['learning_rate'], 
    random_state=RANDOM_STATE,
    scale_pos_weight=19,
    eval_metric='logloss'
)

pipeline_xgb_adults_opt, acc_xgb_adults_opt, prec_xgb_adults_opt, rec_xgb_adults_opt, f1_xgb_adults_opt, auc_xgb_adults_opt, diff_acc_xgb_adults_opt, diff_f1_xgb_adults_opt = train_xgboost_with_pipeline(
    X_train_adults, X_test_adults, y_train_adults, y_test_adults,
    xgb_model_optimized_adults,
    dataset_version="adults"
)


=== XGBoost | adults ===
Train Acc: 1.0000 | Test Acc: 0.9241 | Diff: 0.0759
Train F1:  1.0000 | Test F1:  0.9171 | Diff: 0.0829
⚠ Posible overfitting
Accuracy: 0.9241 | Precision: 0.9112 | Recall: 0.9241 | F1: 0.9171 | AUC: 0.7800


### HACEMOS MATRIZ DE CONFUSION

In [40]:
y_pred_xgb_adults_opt = pipeline_xgb_adults_opt.predict(X_test_adults)
cm_xgb_adults_opt = confusion_matrix(y_test_adults, y_pred_xgb_adults_opt)
plt.figure(figsize=(8, 6))  
sns.heatmap(cm_xgb_adults_opt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],    
            yticklabels=['No Stroke', 'Stroke'])

plt.title('Matriz de Confusión - XGBoost ADULTS (OPTIMIZADO)')
plt.ylabel('Actual')    
plt.xlabel('Predicción')
plt.tight_layout()  
cm_xgb_adults_opt_path = os.path.join(assets_dir, "confusion_matrix_xgb_adults_opt.png")
plt.savefig(cm_xgb_adults_opt_path, dpi=100, bbox_inches='tight')       
plt.close()
print(f"✓ Matriz XGBoost ADULTS (OPTIMIZADO) guardada en: {cm_xgb_adults_opt_path}")       

✓ Matriz XGBoost ADULTS (OPTIMIZADO) guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_adults_opt.png


In [41]:
with mlflow.start_run(run_name="XGBoost_adults_optimized_v2"):
    # Parámetros
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "adults")
    mlflow.log_param("n_estimators", best_params_xgb_adults['n_estimators'])
    mlflow.log_param("max_depth", best_params_xgb_adults['max_depth'])
    mlflow.log_param("learning_rate", best_params_xgb_adults['learning_rate'])
    mlflow.log_param("scale_pos_weight", 19)
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_adults_opt, 4))
    mlflow.log_metric("precision", round(prec_xgb_adults_opt, 4))
    mlflow.log_metric("recall", round(rec_xgb_adults_opt, 4))
    mlflow.log_metric("f1", round(f1_xgb_adults_opt, 4))
    mlflow.log_metric("auc", round(auc_xgb_adults_opt, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_adults_opt, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_adults_opt, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_adults_opt_path, artifact_path="confusion_matrices")
    
     # Modelo
    mlflow.sklearn.log_model(pipeline_xgb_adults_opt, "modelo_xgboost_adults_optimized") 
    
    
    
    print("✓ XGBoost ADULTS OPTIMIZADO registrado (métricas + gráfico)")           


2026/04/21 13:10:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/21 13:10:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ XGBoost ADULTS OPTIMIZADO registrado (métricas + gráfico)
🏃 View run XGBoost_adults_optimized_v2 at: http://192.168.1.152:5001/#/experiments/1/runs/98e8244148bd4db097ea39ff7e681298
🧪 View experiment at: http://192.168.1.152:5001/#/experiments/1


# EXPERIMENTO 
Optimización de XGBoost: Priorizando la vida del paciente (Reducción de Falsos Negativos)
1. El Problema Original: El engaño de las métricas globales
En nuestros primeros intentos, le pedimos a Optuna que optimizara el F1-Score ponderado (average='weighted'). Como nuestro dataset está fuertemente desbalanceado (~95% pacientes sanos y ~5% con ictus), el algoritmo descubrió una "trampa": si predecía que casi todo el mundo estaba sano, conseguía una métrica altísima. Esto nos daba un Accuracy excelente, pero un desastre a nivel médico: muchos Falsos Negativos (pacientes con ictus que el modelo mandaba a casa diagnosticados como "sanos").

2. El Cambio de Enfoque: Mirar solo a la clase "Stroke"
Para solucionar esto, cambiamos la función objetivo de Optuna para que dejara de mirar la media global y se centrara exclusivamente en evaluar a la clase minoritaria (pos_label=1).

3. ¿Por qué usar F2-Score en lugar de F1-Score o Recall puro?
Necesitábamos una métrica que reflejara nuestro objetivo de negocio (o en este caso, médico). Teníamos tres opciones:

Opción A (Recall puro): Si le pedimos al modelo que solo maximice el Recall (atrapar todos los ictus), el modelo se vuelve "vago" y predice que absolutamente todos los pacientes de la base de datos van a tener un ictus. Lograíamos un 100% de Recall, pero saturaríamos el hospital con Falsos Positivos (falsas alarmas). Es inviable.

Opción B (F1-Score): Busca un equilibrio perfecto (50/50) entre atrapar los ictus (Recall) y no dar falsas alarmas (Precision). Mejoró mucho el modelo, pero en medicina un equilibrio 50/50 no siempre es lo ideal.

Opción C (F2-Score - NUESTRA ELECCIÓN): Utilizamos la métrica fbeta_score con beta=2. Esta métrica hace exactamente lo mismo que el F1, pero le da el doble de peso e importancia al Recall que a la Precision.

Conclusión Médica y de Negocio
Al optimizar el F2-Score de la clase 1, le hemos dado al algoritmo de Machine Learning la siguiente instrucción médica: "Te doy permiso para que des más falsas alarmas y seas más paranoico (suben los Falsos Positivos), siempre y cuando eso nos sirva para no dejar escapar a pacientes que realmente están en peligro (minimizar los Falsos Negativos)".

En el mundo real, someter a un paciente sano a unas pruebas preventivas extra tiene un coste asumible; pero diagnosticar como "sano" a un paciente que va a sufrir un infarto cerebral tiene un coste inaceptable.

In [29]:
from sklearn.metrics import recall_score, f1_score

def objective_xgboost_optimized_v2(trial, X_train, X_test, y_train, y_test, dataset_version):
    """
    Función objetivo para Optuna - XGBoost
    OBJETIVO REAL: Maximizar Recall o F1 de la clase minoritaria (Stroke = 1)
    """
    
    # Parámetros a OPTIMIZAR
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 6) 
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2)
    subsample = trial.suggest_float('subsample', 0.7, 1.0)
    
    # Crear modelo
    xgb_model_trial = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        random_state=RANDOM_STATE,
        scale_pos_weight=19,
        eval_metric='logloss'
    )
    
    # Entrenar recuperando el pipeline
    pipeline, _, _, _, _, _, _, _ = train_xgboost_with_pipeline(
        X_train, X_test, y_train, y_test,
        xgb_model_trial,
        dataset_version=dataset_version
    )
    
    # 1. Hacemos las predicciones
    y_pred = pipeline.predict(X_test)
    
    # 2. Devolvemos el F1-Score solo para los Ictus
    return f1_score(y_test, y_pred, pos_label=1)

In [30]:
# Optimizar XGBoost FULL (NEW VERSION)
study_xgb_full_v2 = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

print("🔍 Optimizando XGBoost FULL (v2: RECALL + sin overfitting)...")
study_xgb_full_v2.optimize(
    lambda trial: objective_xgboost_optimized_v2(trial, X_train_full, X_test_full, y_train_full, y_test_full, "full"),
    n_trials=20,
    show_progress_bar=True
)

best_params_xgb_full_v2 = study_xgb_full_v2.best_trial.params
print(f"\n✓ Mejores parámetros XGBoost FULL (v2):")
for param, value in best_params_xgb_full_v2.items():
    print(f"  {param}: {value}")

[I 2026-04-22 12:56:49,708] A new study created in memory with name: no-name-657c0559-f04f-4bec-ba8c-f1fae3eff134


🔍 Optimizando XGBoost FULL (v2: RECALL + sin overfitting)...


  0%|          | 0/20 [00:00<?, ?it/s]


=== XGBoost | full ===


Best trial: 0. Best value: 0.222222:   5%|▌         | 1/20 [00:00<00:06,  2.84it/s]

...

Train Acc: 0.9912 | Test Acc: 0.9198 | Diff: 0.0715
Train F1:  0.9916 | Test F1:  0.9219 | Diff: 0.0696
⚠ Posible overfitting
Accuracy: 0.9198 | Precision: 0.9241 | Recall: 0.9198 | F1: 0.9219 | AUC: 0.8007
[I 2026-04-22 12:56:55,522] Trial 17 finished with value: 0.24528301886792453 and parameters: {'n_estimators': 121, 'max_depth': 6, 'learning_rate': 0.174108982747854, 'subsample': 0.7231389828370709}. Best is trial 6 with value: 0.2564102564102564.

=== XGBoost | full ===


Best trial: 6. Best value: 0.25641:  95%|█████████▌| 19/20 [00:06<00:00,  2.66it/s]

Train Acc: 0.9927 | Test Acc: 0.9157 | Diff: 0.0770
Train F1:  0.9930 | Test F1:  0.9157 | Diff: 0.0772
⚠ Posible overfitting
Accuracy: 0.9157 | Precision: 0.9157 | Recall: 0.9157 | F1: 0.9157 | AUC: 0.7959
[I 2026-04-22 12:56:55,886] Trial 18 finished with value: 0.16 and parameters: {'n_estimators': 123, 'max_depth': 6, 'learning_rate': 0.1990736773545571, 'subsample': 0.710355427833764}. Best is trial 6 with value: 0.2564102564102564.

=== XGBoost | full ===


Best trial: 6. Best value: 0.25641: 100%|██████████| 20/20 [00:06<00:00,  3.12it/s]

Train Acc: 0.8562 | Test Acc: 0.8124 | Diff: 0.0437
Train F1:  0.8927 | Test F1:  0.8600 | Diff: 0.0328
✓ Sin overfitting
Accuracy: 0.8124 | Precision: 0.9311 | Recall: 0.8124 | F1: 0.8600 | AUC: 0.8185
[I 2026-04-22 12:56:56,106] Trial 19 finished with value: 0.23045267489711935 and parameters: {'n_estimators': 77, 'max_depth': 4, 'learning_rate': 0.1694309438253159, 'subsample': 0.7009425811306094}. Best is trial 6 with value: 0.2564102564102564.

✓ Mejores parámetros XGBoost FULL (v2):
  n_estimators: 118
  max_depth: 6
  learning_rate: 0.047938018610088354
  subsample: 0.8542703315240835


In [31]:
xgb_model_optimized_full_v2 = xgb.XGBClassifier(
    n_estimators=best_params_xgb_full_v2['n_estimators'],  
    max_depth= best_params_xgb_full_v2['max_depth'],            
    learning_rate=best_params_xgb_full_v2['learning_rate'], 
    subsample=best_params_xgb_full_v2['subsample'],
    random_state=RANDOM_STATE,
    scale_pos_weight=19,
    eval_metric='logloss'
)

pipeline_xgb_full_opt_v2, acc_xgb_full_opt_v2, prec_xgb_full_opt_v2, rec_xgb_full_opt_v2, f1_xgb_full_opt_v2, auc_xgb_full_opt_v2, diff_acc_xgb_full_opt_v2, diff_f1_xgb_full_opt_v2 = train_xgboost_with_pipeline(
    X_train_full, X_test_full, y_train_full, y_test_full,
    xgb_model_optimized_full_v2,
    dataset_version="full"
)


=== XGBoost | full ===
Train Acc: 0.8725 | Test Acc: 0.8255 | Diff: 0.0470
Train F1:  0.9037 | Test F1:  0.8688 | Diff: 0.0349
✓ Sin overfitting
Accuracy: 0.8255 | Precision: 0.9347 | Recall: 0.8255 | F1: 0.8688 | AUC: 0.8196


In [32]:
y_pred_xgb_full_opt_v2 = pipeline_xgb_full_opt_v2.predict(X_test_full)
cm_xgb_full_opt_v2 = confusion_matrix(y_test_full, y_pred_xgb_full_opt_v2)
plt.figure(figsize=(8, 6))  
sns.heatmap(cm_xgb_full_opt_v2, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],    
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - XGBoost FULL (OPTIMIZADO v2)')
plt.ylabel('Actual')        
plt.xlabel('Predicción')
plt.tight_layout()
cm_xgb_full_opt_v2_path = os.path.join(assets_dir, "confusion_matrix_xgb_full_opt_v2_final.png")
plt.savefig(cm_xgb_full_opt_v2_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✓ Matriz XGBoost FULL (OPTIMIZADO v2) guardada en: {cm_xgb_full_opt_v2_path}") 

✓ Matriz XGBoost FULL (OPTIMIZADO v2) guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_full_opt_v2_final.png


In [46]:
mlflow.end_run()

In [33]:
with mlflow.start_run(run_name="XGBoost_full_optimized_v2_final"):
    # Parámetros    
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("n_estimators", best_params_xgb_full_v2['n_estimators'])
    mlflow.log_param("max_depth", best_params_xgb_full_v2['max_depth'])
    mlflow.log_param("learning_rate", best_params_xgb_full_v2['learning_rate'])
    mlflow.log_param("subsample", best_params_xgb_full_v2['subsample'])
    mlflow.log_param("scale_pos_weight", 19)            
    mlflow.log_param("author", "Gema")  

    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_full_opt_v2, 4))
    mlflow.log_metric("precision", round(prec_xgb_full_opt_v2, 4))
    mlflow.log_metric("recall", round(rec_xgb_full_opt_v2, 4))
    mlflow.log_metric("f1", round(f1_xgb_full_opt_v2, 4))   
    mlflow.log_metric("auc", round(auc_xgb_full_opt_v2, 4))

    # Métricas OVERFITTING
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_full_opt_v2, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_full_opt_v2, 4))     
   
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_full_opt_v2_path, artifact_path="confusion_matrices")    
    
    # Modelo
    mlflow.sklearn.log_model(pipeline_xgb_full_opt_v2, "modelo_xgboost_full_optimized_v2")    
          
    print("✓ XGBoost FULL OPTIMIZADO v2 registrado (métricas + gráfico)")



                                                          

2026/04/22 12:58:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/22 12:58:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ XGBoost FULL OPTIMIZADO v2 registrado (métricas + gráfico)


In [34]:
from sklearn.metrics import fbeta_score # <-- IMPORTANTE: Importamos fbeta_score

def objective_xgboost_optimized_v3(trial, X_train, X_test, y_train, y_test, dataset_version):
    """
    Función objetivo para Optuna - XGBoost
    OBJETIVO REAL: Maximizar el F2-Score (Prioridad doble al Recall de Stroke = 1)
    """
    
    # Parámetros a OPTIMIZAR
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 6) 
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2)
    subsample = trial.suggest_float('subsample', 0.7, 1.0)
    
    # Crear modelo
    xgb_model_trial = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        random_state=RANDOM_STATE,
        scale_pos_weight=19,
        eval_metric='logloss'
    )
    
    # Entrenar recuperando el pipeline
    pipeline, _, _, _, _, _, _, _ = train_xgboost_with_pipeline(
        X_train, X_test, y_train, y_test,
        xgb_model_trial,
        dataset_version=dataset_version
    )
    
    # 1. Hacemos las predicciones
    y_pred = pipeline.predict(X_test)
    
    # 2. Devolvemos el F2-Score solo para los Ictus (beta=2 da prioridad al Recall)
    return fbeta_score(y_test, y_pred, beta=2, pos_label=1) # <-- EL CAMBIO ESTÁ AQUÍ

In [35]:
# Optimizar XGBoost FULL (NEW VERSION)
study_xgb_full_v3 = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

print("🔍 Optimizando XGBoost FULL (v3: F2-Score)...")
study_xgb_full_v3.optimize(
    lambda trial: objective_xgboost_optimized_v3(trial, X_train_full, X_test_full, y_train_full, y_test_full, "full"),
    n_trials=20,
    show_progress_bar=True
)

best_params_xgb_full_v3 = study_xgb_full_v3.best_trial.params
print(f"\n✓ Mejores parámetros XGBoost FULL (v3):")
for param, value in best_params_xgb_full_v3.items():
    print(f"  {param}: {value}")

[I 2026-04-22 12:58:58,344] A new study created in memory with name: no-name-2874d6fb-026d-434c-b5a8-53539f65006f


🔍 Optimizando XGBoost FULL (v3: F2-Score)...


  0%|          | 0/20 [00:00<?, ?it/s]


=== XGBoost | full ===


Best trial: 0. Best value: 0.253623:   5%|▌         | 1/20 [00:00<00:08,  2.16it/s]

...

Train Acc: 0.7982 | Test Acc: 0.7733 | Diff: 0.0249
Train F1:  0.8537 | Test F1:  0.8354 | Diff: 0.0183
✓ Sin overfitting
Accuracy: 0.7733 | Precision: 0.9394 | Recall: 0.7733 | F1: 0.8354 | AUC: 0.8214
[I 2026-04-22 12:59:03,411] Trial 17 finished with value: 0.4017857142857143 and parameters: {'n_estimators': 149, 'max_depth': 5, 'learning_rate': 0.034127120360206514, 'subsample': 0.7456601487309656}. Best is trial 17 with value: 0.4017857142857143.

=== XGBoost | full ===


Best trial: 17. Best value: 0.401786:  95%|█████████▌| 19/20 [00:05<00:00,  3.36it/s]

Train Acc: 0.7204 | Test Acc: 0.7242 | Diff: 0.0038
Train F1:  0.7994 | Test F1:  0.8017 | Diff: 0.0023
✓ Sin overfitting
Accuracy: 0.7242 | Precision: 0.9384 | Recall: 0.7242 | F1: 0.8017 | AUC: 0.8134
[I 2026-04-22 12:59:03,701] Trial 18 finished with value: 0.37074148296593185 and parameters: {'n_estimators': 126, 'max_depth': 5, 'learning_rate': 0.028213024211565088, 'subsample': 0.7335928350994433}. Best is trial 17 with value: 0.4017857142857143.

=== XGBoost | full ===


Best trial: 17. Best value: 0.401786: 100%|██████████| 20/20 [00:05<00:00,  3.52it/s]

Train Acc: 0.9270 | Test Acc: 0.8726 | Diff: 0.0543
Train F1:  0.9410 | Test F1:  0.8946 | Diff: 0.0463
⚠ Posible overfitting
Accuracy: 0.8726 | Precision: 0.9222 | Recall: 0.8726 | F1: 0.8946 | AUC: 0.8127
[I 2026-04-22 12:59:04,024] Trial 19 finished with value: 0.2733118971061093 and parameters: {'n_estimators': 144, 'max_depth': 5, 'learning_rate': 0.10128104801813886, 'subsample': 0.8662580266913333}. Best is trial 17 with value: 0.4017857142857143.

✓ Mejores parámetros XGBoost FULL (v3):
  n_estimators: 149
  max_depth: 5
  learning_rate: 0.034127120360206514
  subsample: 0.7456601487309656


In [36]:
xgb_model_optimized_full_v3 = xgb.XGBClassifier(
    n_estimators=best_params_xgb_full_v3['n_estimators'],  
    max_depth= best_params_xgb_full_v3['max_depth'],            
    learning_rate=best_params_xgb_full_v3['learning_rate'], 
    subsample=best_params_xgb_full_v3['subsample'],
    random_state=RANDOM_STATE,
    scale_pos_weight=19,
    eval_metric='logloss'
)

pipeline_xgb_full_opt_v3, acc_xgb_full_opt_v3, prec_xgb_full_opt_v3, rec_xgb_full_opt_v3, f1_xgb_full_opt_v3, auc_xgb_full_opt_v3, diff_acc_xgb_full_opt_v3, diff_f1_xgb_full_opt_v3 = train_xgboost_with_pipeline(
    X_train_full, X_test_full, y_train_full, y_test_full,
    xgb_model_optimized_full_v3,
    dataset_version="full"
)


=== XGBoost | full ===
Train Acc: 0.7982 | Test Acc: 0.7733 | Diff: 0.0249
Train F1:  0.8537 | Test F1:  0.8354 | Diff: 0.0183
✓ Sin overfitting
Accuracy: 0.7733 | Precision: 0.9394 | Recall: 0.7733 | F1: 0.8354 | AUC: 0.8214


In [37]:
y_pred_xgb_full_opt_v3 = pipeline_xgb_full_opt_v3.predict(X_test_full)
cm_xgb_full_opt_v3 = confusion_matrix(y_test_full, y_pred_xgb_full_opt_v3)

plt.figure(figsize=(8, 6))  
sns.heatmap(cm_xgb_full_opt_v3, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],    
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - XGBoost FULL (OPTIMIZADO v3)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()
cm_xgb_full_opt_v3_path = os.path.join(assets_dir, "confusion_matrix_xgb_full_opt_v3_final.png")

plt.savefig(cm_xgb_full_opt_v3_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✓ Matriz XGBoost FULL (OPTIMIZADO v3) guardada en: {cm_xgb_full_opt_v3_path}") 

      

✓ Matriz XGBoost FULL (OPTIMIZADO v3) guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_full_opt_v3_final.png


In [38]:
with mlflow.start_run(run_name="XGBoost_full_optimized_v3_final"):
    # Parámetros    
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("n_estimators", best_params_xgb_full_v3['n_estimators'])
    mlflow.log_param("max_depth", best_params_xgb_full_v3['max_depth'])
    mlflow.log_param("learning_rate", best_params_xgb_full_v3['learning_rate'])
    mlflow.log_param("subsample", best_params_xgb_full_v3['subsample'])
    mlflow.log_param("scale_pos_weight", 19)            
    mlflow.log_param("author", "Gema")  

    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_full_opt_v3, 4))
    mlflow.log_metric("precision", round(prec_xgb_full_opt_v3, 4))
    mlflow.log_metric("recall", round(rec_xgb_full_opt_v3, 4))
    mlflow.log_metric("f1", round(f1_xgb_full_opt_v3, 4))   
    mlflow.log_metric("auc", round(auc_xgb_full_opt_v3, 4))

    # Métricas OVERFITTING
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_full_opt_v3, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_full_opt_v3, 4))     
   
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_full_opt_v3_path, artifact_path="confusion_matrices")    
    
    # Modelo
    mlflow.sklearn.log_model(pipeline_xgb_full_opt_v3, "modelo_xgboost_full_optimized_v3")    
          
    print("✓ XGBoost FULL OPTIMIZADO v3 registrado (métricas + gráfico)")


2026/04/22 12:59:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/22 12:59:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ XGBoost FULL OPTIMIZADO v3 registrado (métricas + gráfico)
